### Importing Libraries

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
BATCH_SIZE = 64
RANDOM_SEED = 42
NUM_EPOCHS = 20
DEVICE = 'cuda:0'

In [5]:
lenet_transforms = transforms.Compose([
    transforms.Resize((32, 32)),    
    transforms.ToTensor(),        
    transforms.Normalize((0.1307,), (0.3081,))
])

In [6]:
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=lenet_transforms
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    transform=lenet_transforms
)

In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)

images, labels = next(iter(train_loader))
print("Image batch dimensions:", images.shape)
print("Image label dimensions:", labels.shape)

Image batch dimensions: torch.Size([64, 1, 32, 32])
Image label dimensions: torch.Size([64])


#### Custom LeCun "SigmoidLike" Activation

$$
\begin{aligned}
x_{i} &= f(a_{i})\\
f(a) &= A \tanh (Sa)
\end{aligned}
$$

In [12]:
class SigmoidLike(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.A = 1.17159
        self.S = 2/3
        
    def forward(self, x):
        return self.A  * torch.tanh(self.S * x)

### Custom C3 Layer


In [ ]:
class LeCunC3(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        C3_CONNECTION_MAP = [
            [0,1,2],
            [1,2,3],
            [2,3,4],
            [3,4,5],
            [0,4,5],
            [0,1,5],
            [0,1,2,3],
            [1,2,3,4],
            [2,3,4,5],
            [0,3,4,5],
            [0,1,4,5],
            [0,1,2,5],
            [0,1,3,4],
            [1,2,4,5],
            [0,2,3,5],
            [0,1,2,3,4,5]
        ]

        self.c3_featuremaps = nn.ModuleList([
            nn.Conv2d(in_channels=len(inputs),out_channels=1, kernel=5)
            for inputs in C3_CONNECTION_MAP
        ])

    def forward(self,x):
        outputs = []
        for conv, input_maps in zip(self.c3_featuremaps, C3_CONNECTION_MAP):
            x_subset = x[:, input_maps, :, :]
            outputs.append(conv(x_subset))

        return outputs

## Model

In [17]:
torch.manual_seed(RANDOM_SEED)
class LeNet5(nn.Module):
    def __init__(self, num_classes, activation = nn.Tanh):
        super().__init__()
        
        self.features = torch.nn.Sequential(
            nn.Conv2d(1,6,5),
            activation(),
            nn.AvgPool2d(2),
            nn.Conv2d(6,16,5),
            activation(),
            nn.AvgPool2d(2),
        )    
        
        self.head = torch.nn.Sequential(
            nn.Conv2d(16,120,5), ## SWAP LATER AND CHECK
            activation(),
            nn.Flatten(),
            nn.Linear(120,84),
            activation(),
            nn.Linear(84,num_classes)
        )

    
    def forward(self, x):
        x = self.features(x)
        logits = self.head(x)
        probas = torch.nn.functional.softmax(input=logits,dim=1)
        return logits, probas

### Model Init

In [21]:
model = LeNet5(num_classes=len(test_loader.dataset.targets.unique()))
model.to(device="cuda")

LeNet5(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (head): Sequential(
    (0): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): Flatten(start_dim=1, end_dim=-1)
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): Tanh()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)

In [22]:
optim = torch.optim.SGD(model.parameters(),lr=0.1)

## Model Training

In [29]:
for epoch in range(NUM_EPOCHS):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
       features = features.to(DEVICE)
       targets = targets.to(DEVICE)
       
       #Forward
       logits, probas = model(features)
       cost = nn.functional.cross_entropy(logits, targets)
       
       optim.zero_grad()
       cost.backward()
       
       optim.step()
       
       #Logging
       print(f"Epoch : {epoch+1} | Batch: {batch_idx} | Cost: {cost}")
       
    with torch.inference_mode():
        pass
        

Epoch : 1 | Batch: 0 | Cost: 2.2958240509033203
Epoch : 1 | Batch: 1 | Cost: 2.2872376441955566
Epoch : 1 | Batch: 2 | Cost: 2.2837202548980713
Epoch : 1 | Batch: 3 | Cost: 2.2885618209838867
Epoch : 1 | Batch: 4 | Cost: 2.263361692428589
Epoch : 1 | Batch: 5 | Cost: 2.2424795627593994
Epoch : 1 | Batch: 6 | Cost: 2.2254817485809326
Epoch : 1 | Batch: 7 | Cost: 2.2230095863342285
Epoch : 1 | Batch: 8 | Cost: 2.249265670776367
Epoch : 1 | Batch: 9 | Cost: 2.1907150745391846
Epoch : 1 | Batch: 10 | Cost: 2.1574573516845703
Epoch : 1 | Batch: 11 | Cost: 2.166900873184204
Epoch : 1 | Batch: 12 | Cost: 2.1300199031829834
Epoch : 1 | Batch: 13 | Cost: 2.1123604774475098
Epoch : 1 | Batch: 14 | Cost: 2.0921831130981445
Epoch : 1 | Batch: 15 | Cost: 2.088205337524414
Epoch : 1 | Batch: 16 | Cost: 2.0808801651000977
Epoch : 1 | Batch: 17 | Cost: 1.9782450199127197
Epoch : 1 | Batch: 18 | Cost: 1.9454463720321655
Epoch : 1 | Batch: 19 | Cost: 2.0074892044067383
Epoch : 1 | Batch: 20 | Cost: 1.92

KeyboardInterrupt: 